### Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

### Generate Customer IDs

In [3]:
customer_ids = [
    f"C{i:05d}"
    for i in range(1, 5001)
]

In [4]:
customer_ids[:10]

['C00001',
 'C00002',
 'C00003',
 'C00004',
 'C00005',
 'C00006',
 'C00007',
 'C00008',
 'C00009',
 'C00010']

### Generate Gender

In [5]:
genders = np.random.choice(
    ["Male", "Female"],
    size=5000,
    p=[0.55, 0.45]
)

In [6]:
pd.Series(genders).value_counts()

Male      2748
Female    2252
Name: count, dtype: int64

### Generate Age

In [7]:
age_groups = np.random.choice(
    ["18-24", "25-34", "35-44", "45-54", "55+"],
    size=5000,
    p=[0.15, 0.30, 0.28, 0.17, 0.10]
)

In [8]:
age_ranges = {
    "18-24": (18, 24),
    "25-34": (25, 34),
    "35-44": (35, 44),
    "45-54": (45, 54),
    "55+": (55, 70)
}

In [9]:
ages = [
    np.random.randint(
        age_ranges[group][0],
        age_ranges[group][1] + 1
    )
    for group in age_groups
]

In [10]:
pd.Series(ages).describe()

count    5000.000000
mean       37.373600
std        12.412608
min        18.000000
25%        28.000000
50%        36.000000
75%        45.000000
max        70.000000
dtype: float64

In [11]:
pd.cut(
    ages,
    bins=[17, 24, 34, 44, 54, 70],
    labels=["18-24", "25-34", "35-44", "45-54", "55+"]
).value_counts().sort_index()

18-24     759
25-34    1546
35-44    1429
45-54     784
55+       482
Name: count, dtype: int64

### Customer Cities

In [12]:
cities = [
    "Riyadh",
    "Jeddah",
    "Dammam",
    "Khobar",
    "Makkah",
    "Madinah",
    "Abha",
    "Tabuk"
]

In [13]:
city_probabilities = [
    0.30,  # Riyadh
    0.22,  # Jeddah
    0.12,  # Dammam
    0.08,  # Khobar
    0.10,  # Makkah
    0.08,  # Madinah
    0.06,  # Abha
    0.04   # Tabuk
]

In [14]:
customer_cities = np.random.choice(
    cities,
    size=5000,
    p=city_probabilities
)

In [15]:
pd.Series(customer_cities).value_counts()

Riyadh     1446
Jeddah     1101
Dammam      600
Makkah      538
Khobar      412
Madinah     411
Abha        293
Tabuk       199
Name: count, dtype: int64

### Customer Segmentation

In [16]:
segment_names = [
    "Premium",
    "Regular",
    "Occasional",
    "New"
]

segment_probabilities = [
    0.10,
    0.45,
    0.30,
    0.15
]

In [17]:
customer_segments = np.random.choice(
    segment_names,
    size=5000,
    p=segment_probabilities
)

In [18]:
pd.Series(customer_segments).value_counts()

Regular       2169
Occasional    1520
New            809
Premium        502
Name: count, dtype: int64

### Create the DataFrame

In [19]:
df_customer = pd.DataFrame({
    "customer_id": customer_ids,
    "gender": genders,
    "age": ages,
    "city": customer_cities,
    "customer_segment": customer_segments
})

In [20]:
df_customer.head(10)

,customer_id,gender,age,city,customer_segment
0,C00001,Male,31,Riyadh,New
1,C00002,Female,38,Dammam,Regular
2,C00003,Female,46,Khobar,Regular
3,C00004,Female,29,Jeddah,Regular
4,C00005,Male,47,Madinah,Regular
5,C00006,Male,24,Jeddah,Occasional
6,C00007,Male,53,Riyadh,New
7,C00008,Female,50,Madinah,New
8,C00009,Female,25,Madinah,New
9,C00010,Female,27,Riyadh,Regular


### Validate the shape

In [21]:
df_customer.shape

(5000, 5)

### Validate Customer IDs

In [22]:
df_customer["customer_id"].nunique()

5000

In [23]:
df_customer["customer_id"].duplicated().sum()

np.int64(0)

### Validate Missing Values

In [24]:
df_customer.isna().sum()

customer_id         0
gender              0
age                 0
city                0
customer_segment    0
dtype: int64

### Validate Age

In [25]:
df_customer["age"].min()

np.int64(18)

In [26]:
df_customer["age"].max()

np.int64(70)

In [27]:
(df_customer["age"] < 18).sum()
(df_customer["age"] > 70).sum()

np.int64(0)

### Validate Customer Segments

In [28]:
df_customer["customer_segment"].value_counts()

customer_segment
Regular       2169
Occasional    1520
New            809
Premium        502
Name: count, dtype: int64

In [29]:
df_customer["customer_segment"].value_counts(
    normalize=True
).round(3)

customer_segment
Regular       0.434
Occasional    0.304
New           0.162
Premium       0.100
Name: proportion, dtype: float64

### Cross-Check Customer Segments

In [30]:
pd.crosstab(
    df_customer["city"],
    df_customer["customer_segment"],
    normalize="index"
).round(3)

customer_segment,New,Occasional,Premium,Regular
city,,,,
Abha,0.150,0.314,0.068,0.468
Dammam,0.148,0.292,0.117,0.443
Jeddah,0.159,0.304,0.103,0.434
Khobar,0.189,0.318,0.063,0.430
Madinah,0.180,0.287,0.105,0.428
Makkah,0.164,0.296,0.108,0.433
Riyadh,0.156,0.311,0.102,0.432
Tabuk,0.176,0.307,0.126,0.392


### Cross-Check Age and Segment

In [31]:
df_customer.groupby(
    "customer_segment"
)["age"].agg(
    ["count", "mean", "min", "max"]
).round(1)

,count,mean,min,max
customer_segment,,,,
New,809,37.4,18,70
Occasional,1520,37.2,18,70
Premium,502,37.5,18,70
Regular,2169,37.4,18,70


### Save the Customer Master

In [32]:
Path("../data/raw").mkdir(
    parents=True,
    exist_ok=True
)

In [33]:
df_customer.to_csv(
    "../data/raw/dim_customer.csv",
    index=False
)

In [34]:
Path(
    "../data/raw/dim_customer.csv"
).exists()

True

### Final Data Quality Check

In [35]:
validation = {
    "row_count": len(df_customer),
    "unique_customer_ids": df_customer["customer_id"].nunique(),
    "duplicate_customer_ids": df_customer["customer_id"].duplicated().sum(),
    "missing_values": df_customer.isna().sum().sum(),
    "invalid_age_below_18": (df_customer["age"] < 18).sum(),
    "invalid_age_above_70": (df_customer["age"] > 70).sum()
}

validation

{'row_count': 5000,
 'unique_customer_ids': 5000,
 'duplicate_customer_ids': np.int64(0),
 'missing_values': np.int64(0),
 'invalid_age_below_18': np.int64(0),
 'invalid_age_above_70': np.int64(0)}